# PlainScript Dataset EDA

Exploratory data analysis across all training dataset versions (V1, V2, V2.5, V3) and their component sources. Goal: identify opportunities for a V4 dataset that improves simplification quality.

In [ ]:
import json, os, re, statistics, csv
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['figure.dpi'] = 100

BASE = "C:/Users/virtu/Documents/PlainScript/medclear_results"

In [ ]:
# ---- Utility functions ----

def load_jsonl(path, max_lines=None):
    rows = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_lines and i >= max_lines:
                break
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return rows


def detect_keys(sample):
    src_key = next((k for k in ["source", "input", "text", "src", "input_text"] if k in sample), None)
    tgt_key = next((k for k in ["target", "output", "simple", "tgt", "simplified", "target_text"] if k in sample), None)
    return src_key, tgt_key


JARGON = {
    "hypertension", "cholecystectomy", "appendectomy", "laparoscopic",
    "intraoperative", "postoperative", "hemorrhage", "edema",
    "tachycardia", "bradycardia", "dyspnea", "stenosis",
    "bilateral", "unilateral", "prophylaxis", "analgesic",
    "pathology", "etiology", "prognosis", "differential",
    "hematoma", "embolism", "ischemia", "thrombosis",
    "perfusion", "intubation", "catheterization", "angiography",
    "effusion", "pneumothorax", "fibrillation", "ablation",
    "debridement", "excision", "resection", "anastomosis",
    "hemoglobin", "creatinine", "bilirubin", "troponin",
}


def flesch_kincaid(text):
    """Approximate Flesch-Kincaid grade level."""
    sentences = max(len(re.split(r'[.!?]+', text)), 1)
    words_list = text.split()
    words = max(len(words_list), 1)
    syllables = sum(max(len(re.findall(r'[aeiouy]+', w.lower())), 1) for w in words_list)
    return 0.39 * (words / sentences) + 11.8 * (syllables / words) - 15.59


def analyze_dataset(name, rows):
    """Analyze a dataset and return stats dict."""
    if not rows:
        return {}

    src_key, tgt_key = detect_keys(rows[0])
    if not src_key or not tgt_key:
        print(f"  {name}: keys={list(rows[0].keys())} - could not identify src/tgt")
        return {}

    src_lens, tgt_lens, ratios = [], [], []
    granularities = Counter()
    jargon_retained = 0
    empty_tgt = 0
    duplicates_src = Counter()
    fk_scores = []

    for row in rows:
        src = str(row.get(src_key, ""))
        tgt = str(row.get(tgt_key, ""))
        src_clean = re.sub(r'^(simplify medical|simplify|define medical term)[:\s]*', '', src).strip()

        src_words = len(src_clean.split())
        tgt_words = len(tgt.split())
        src_lens.append(src_words)
        tgt_lens.append(tgt_words)

        if src_words > 0:
            ratios.append(tgt_words / src_words)
        if tgt_words == 0:
            empty_tgt += 1

        if src_words <= 5:
            granularities["term"] += 1
        elif src_words <= 15:
            granularities["phrase"] += 1
        elif src_words <= 40:
            granularities["sentence"] += 1
        else:
            granularities["paragraph"] += 1

        if any(j in tgt.lower() for j in JARGON):
            jargon_retained += 1

        if len(fk_scores) < 2000 and tgt_words > 5:
            fk_scores.append(flesch_kincaid(tgt))

        duplicates_src[src_clean.lower().strip()[:100]] += 1

    n = len(src_lens)
    if n == 0:
        return {}

    dup_total = sum(v - 1 for v in duplicates_src.values() if v > 1)

    return {
        "name": name, "n": n,
        "src_mean": statistics.mean(src_lens),
        "src_median": statistics.median(src_lens),
        "tgt_mean": statistics.mean(tgt_lens),
        "tgt_median": statistics.median(tgt_lens),
        "ratio_mean": statistics.mean(ratios),
        "ratio_median": statistics.median(ratios),
        "jargon_pct": jargon_retained / n * 100,
        "dup_pct": dup_total / n * 100,
        "fk_mean": statistics.mean(fk_scores) if fk_scores else 0,
        "fk_median": statistics.median(fk_scores) if fk_scores else 0,
        "granularities": dict(granularities),
        "src_lens": src_lens,
        "tgt_lens": tgt_lens,
        "fk_scores": fk_scores,
    }

## Section 1: Training Set Versions (Head-to-Head)

Compare all four training dataset versions across key quality metrics.

In [ ]:
datasets = {
    "V1 (original)": f"{BASE}/training_final/train.jsonl",
    "V2 (deployed)": f"{BASE}/training_final_v2/train.jsonl",
    "V2.5": f"{BASE}/training_final_v25/train.jsonl",
    "V3 (large)": f"{BASE}/training_final_v3/train.jsonl",
}

results = {}
for name, path in datasets.items():
    if os.path.exists(path):
        rows = load_jsonl(path)
        results[name] = analyze_dataset(name, rows)
        print(f"{name}: {results[name]['n']:,} examples loaded")

In [ ]:
# Summary comparison table
print(f"{'Dataset':<20} {'N':>8} {'Src':>6} {'Tgt':>6} {'Ratio':>6} {'Jargon%':>8} {'FK':>5} {'Dup%':>6}")
print(f"{'-'*18:<20} {'-'*8:>8} {'-'*6:>6} {'-'*6:>6} {'-'*6:>6} {'-'*8:>8} {'-'*5:>5} {'-'*6:>6}")
for name, r in results.items():
    if r:
        print(f"{name:<20} {r['n']:>8,} {r['src_mean']:>6.1f} {r['tgt_mean']:>6.1f} "
              f"{r['ratio_mean']:>6.2f} {r['jargon_pct']:>7.1f}% {r['fk_mean']:>5.1f} {r['dup_pct']:>5.1f}%")

In [ ]:
# --- Granularity distribution: stacked bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))

labels = list(results.keys())
categories = ["term", "phrase", "sentence", "paragraph"]
colors = ["#4CAF50", "#2196F3", "#FF9800", "#F44336"]

bottoms = [0] * len(labels)
for cat, color in zip(categories, colors):
    vals = []
    for name in labels:
        r = results[name]
        total = r["n"]
        vals.append(r["granularities"].get(cat, 0) / total * 100)
    ax.bar(labels, vals, bottom=bottoms, label=cat, color=color)
    bottoms = [b + v for b, v in zip(bottoms, vals)]

ax.set_ylabel("% of training examples")
ax.set_title("Granularity Distribution by Dataset Version")
ax.legend(loc="upper right")
ax.set_ylim(0, 105)
for i, name in enumerate(labels):
    ax.text(i, 102, f"n={results[name]['n']:,}", ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Quality metrics comparison: jargon leak, FK grade, duplicate rate ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = list(results.keys())
x = range(len(names))

# Jargon leak
vals = [results[n]["jargon_pct"] for n in names]
bars = axes[0].bar(x, vals, color=["#4CAF50" if v < 5 else "#FF9800" if v < 10 else "#F44336" for v in vals])
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel("%")
axes[0].set_title("Jargon Leak in Targets")
axes[0].axhline(y=5, color='green', linestyle='--', alpha=0.5, label='5% threshold')
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{v:.1f}%", ha='center', fontsize=9)

# FK grade
vals = [results[n]["fk_mean"] for n in names]
bars = axes[1].bar(x, vals, color=["#4CAF50" if v < 9 else "#FF9800" if v < 11 else "#F44336" for v in vals])
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=20, ha='right', fontsize=9)
axes[1].set_ylabel("Grade level")
axes[1].set_title("Flesch-Kincaid Grade (Target)")
axes[1].axhline(y=8, color='green', linestyle='--', alpha=0.5, label='8th grade target')
for bar, v in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, f"{v:.1f}", ha='center', fontsize=9)

# Duplicate rate
vals = [results[n]["dup_pct"] for n in names]
bars = axes[2].bar(x, vals, color=["#4CAF50" if v < 10 else "#FF9800" if v < 30 else "#F44336" for v in vals])
axes[2].set_xticks(x)
axes[2].set_xticklabels(names, rotation=20, ha='right', fontsize=9)
axes[2].set_ylabel("%")
axes[2].set_title("Duplicate Source Rate")
for bar, v in zip(bars, vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{v:.1f}%", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# --- Source vs Target length distributions ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, (name, r) in enumerate(results.items()):
    ax = axes[idx // 2][idx % 2]
    src_clip = [min(s, 150) for s in r["src_lens"]]
    tgt_clip = [min(t, 150) for t in r["tgt_lens"]]
    ax.hist(src_clip, bins=50, alpha=0.6, label="Source", color="#2196F3")
    ax.hist(tgt_clip, bins=50, alpha=0.6, label="Target", color="#FF9800")
    ax.set_title(f"{name} (n={r['n']:,})")
    ax.set_xlabel("Word count (clipped at 150)")
    ax.legend(fontsize=8)
    ax.set_xlim(0, 150)

plt.suptitle("Source vs Target Length Distributions", fontsize=13)
plt.tight_layout()
plt.show()

## Section 2: V3 Components (What Went Wrong)

V3 had 4.2x more data than V2 but produced worse simplifications. Breaking down each component source to understand why.

In [ ]:
v3_components = {
    "Asclepius": f"{BASE}/v3_data/asclepius_pairs.jsonl",
    "CHV filtered": f"{BASE}/v3_data/chv_pairs_filtered.jsonl",
    "CHV raw": f"{BASE}/v3_data/chv_pairs.jsonl",
    "MedQuAD": f"{BASE}/v3_data/medquad_pairs.jsonl",
    "Liliya": f"{BASE}/v3_data/liliya_pairs.jsonl",
    "Drug pairs": f"{BASE}/v3_data/drug_pairs.jsonl",
    "Synthetic V3": f"{BASE}/v3_data/synthetic_v3.jsonl",
}

v3_results = {}
for name, path in v3_components.items():
    if os.path.exists(path):
        rows = load_jsonl(path, max_lines=5000)
        v3_results[name] = analyze_dataset(name, rows)
        r = v3_results[name]
        if r:
            print(f"{name:20} n={r['n']:>6,}  jargon={r['jargon_pct']:5.1f}%  "
                  f"FK={r['fk_mean']:5.1f}  dup={r['dup_pct']:5.1f}%  ratio={r['ratio_mean']:.2f}")

In [ ]:
# --- V3 component quality comparison ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

v3_names = [n for n in v3_results if v3_results[n]]
x = range(len(v3_names))

# Jargon leak by component
vals = [v3_results[n]["jargon_pct"] for n in v3_names]
bars = axes[0].barh(v3_names, vals, color=["#F44336" if v > 10 else "#FF9800" if v > 5 else "#4CAF50" for v in vals])
axes[0].set_xlabel("Jargon leak %")
axes[0].set_title("V3 Components: Jargon Retained in Target")
axes[0].axvline(x=5, color='green', linestyle='--', alpha=0.5)
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f"{v:.1f}%", va='center', fontsize=9)

# FK grade by component
vals = [v3_results[n]["fk_mean"] for n in v3_names]
bars = axes[1].barh(v3_names, vals, color=["#F44336" if v > 12 else "#FF9800" if v > 9 else "#4CAF50" for v in vals])
axes[1].set_xlabel("FK Grade Level")
axes[1].set_title("V3 Components: Target Reading Level")
axes[1].axvline(x=8, color='green', linestyle='--', alpha=0.5)
for bar, v in zip(bars, vals):
    axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, f"{v:.1f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Section 3: V2 Components (What Worked)

V2 is the deployed model. Understanding which components contributed most to its success.

In [ ]:
v2_components = {
    "Academic chunks": f"{BASE}/training_v2/academic_chunks.jsonl",
    "Agent terms": f"{BASE}/training_v2/agent_terms.jsonl",
    "Extra phrases": f"{BASE}/training_v2/extra_phrases.jsonl",
    "Extra terms": f"{BASE}/training_v2/extra_terms.jsonl",
    "Aligned synthetic": f"{BASE}/synthetic_data/aligned_training_data.jsonl",
    "Claude pairs": f"{BASE}/synthetic_data/claude_generated_pairs.jsonl",
    "Claude COT": f"{BASE}/synthetic_data/claude_generated_pairs_cot.jsonl",
    "Claude RAG": f"{BASE}/synthetic_data/claude_generated_pairs_rag.jsonl",
}

v2_results = {}
for name, path in v2_components.items():
    if os.path.exists(path):
        rows = load_jsonl(path, max_lines=5000)
        v2_results[name] = analyze_dataset(name, rows)
        r = v2_results[name]
        if r:
            print(f"{name:20} n={r['n']:>6,}  jargon={r['jargon_pct']:5.1f}%  "
                  f"FK={r['fk_mean']:5.1f}  dup={r['dup_pct']:5.1f}%  ratio={r['ratio_mean']:.2f}")

In [ ]:
# --- V2 component quality comparison ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

v2_names = [n for n in v2_results if v2_results[n]]

vals = [v2_results[n]["jargon_pct"] for n in v2_names]
bars = axes[0].barh(v2_names, vals, color=["#F44336" if v > 10 else "#FF9800" if v > 5 else "#4CAF50" for v in vals])
axes[0].set_xlabel("Jargon leak %")
axes[0].set_title("V2 Components: Jargon Retained in Target")
axes[0].axvline(x=5, color='green', linestyle='--', alpha=0.5)
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f"{v:.1f}%", va='center', fontsize=9)

vals = [v2_results[n]["fk_mean"] for n in v2_names]
bars = axes[1].barh(v2_names, vals, color=["#F44336" if v > 12 else "#FF9800" if v > 9 else "#4CAF50" for v in vals])
axes[1].set_xlabel("FK Grade Level")
axes[1].set_title("V2 Components: Target Reading Level")
axes[1].axvline(x=8, color='green', linestyle='--', alpha=0.5)
for bar, v in zip(bars, vals):
    axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, f"{v:.1f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Section 4: FK Grade Distribution Deep Dive

The target FK grade determines whether patients can actually read the output. Patient materials should target 6th-8th grade level.

In [ ]:
# --- FK grade distributions for each version ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, (name, r) in enumerate(results.items()):
    ax = axes[idx // 2][idx % 2]
    if r.get("fk_scores"):
        fk_clip = [max(min(f, 25), -5) for f in r["fk_scores"]]
        ax.hist(fk_clip, bins=60, alpha=0.7, color="#2196F3", edgecolor="white")
        ax.axvline(x=8, color='green', linestyle='--', linewidth=2, label='8th grade')
        ax.axvline(x=r["fk_mean"], color='red', linestyle='-', linewidth=2, label=f'mean={r["fk_mean"]:.1f}')
        below_8 = sum(1 for f in r["fk_scores"] if f <= 8) / len(r["fk_scores"]) * 100
        ax.set_title(f"{name} | {below_8:.0f}% at or below 8th grade")
        ax.legend(fontsize=8)
    ax.set_xlabel("FK Grade Level")
    ax.set_xlim(-5, 25)

plt.suptitle("Flesch-Kincaid Grade Level Distribution (Target Text)", fontsize=13)
plt.tight_layout()
plt.show()

## Section 5: V2 vs V3 Direct Comparison

V2 (20,841 examples) outperformed V3 (87,189 examples) on actual simplification quality. Here's the data explaining why.

In [ ]:
if "V2 (deployed)" in results and "V3 (large)" in results:
    v2 = results["V2 (deployed)"]
    v3 = results["V3 (large)"]
    
    print("WHY V3 UNDERPERFORMED V2")
    print("=" * 50)
    print(f"V3 had {v3['n']/v2['n']:.1f}x more data but:")
    print(f"  Jargon leak:  V2={v2['jargon_pct']:.1f}%   V3={v3['jargon_pct']:.1f}%  ({v3['jargon_pct']/max(v2['jargon_pct'],0.1):.1f}x worse)")
    print(f"  FK grade:     V2={v2['fk_mean']:.1f}      V3={v3['fk_mean']:.1f}")
    print(f"  Duplicates:   V2={v2['dup_pct']:.1f}%   V3={v3['dup_pct']:.1f}%")
    print()
    print("V2 granularity (what worked):")
    for g in ["term", "phrase", "sentence", "paragraph"]:
        c = v2["granularities"].get(g, 0)
        print(f"  {g:12}: {c:6,} ({c/v2['n']*100:5.1f}%)")
    print()
    print("V3 granularity (what didn't work):")
    for g in ["term", "phrase", "sentence", "paragraph"]:
        c = v3["granularities"].get(g, 0)
        print(f"  {g:12}: {c:6,} ({c/v3['n']*100:5.1f}%)")

In [ ]:
# --- Side-by-side granularity ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cats = ["term", "phrase", "sentence", "paragraph"]
colors = ["#4CAF50", "#2196F3", "#FF9800", "#F44336"]

for ax_idx, (name, r) in enumerate([("V2 (deployed)", v2), ("V3 (large)", v3)]):
    vals = [r["granularities"].get(c, 0) / r["n"] * 100 for c in cats]
    axes[ax_idx].pie(vals, labels=[f"{c}\n{v:.0f}%" for c, v in zip(cats, vals)],
                     colors=colors, startangle=90)
    axes[ax_idx].set_title(f"{name} (n={r['n']:,})")

plt.suptitle("Granularity: V2 (winner) vs V3", fontsize=13)
plt.tight_layout()
plt.show()

## Section 6: Actionable Findings for V4

### Key Insight
**Data distribution matters more than data quantity.** V2 (20K examples, 67% terms+phrases) beat V3 (87K examples, 35% paragraphs) because:
1. V2 taught vocabulary before composition (curriculum learning)
2. V3's Asclepius data (17.7% jargon leak, FK 12.4) taught paraphrasing at medical-student level, not simplification
3. V2 had 41.6% duplicates -- easy wins from deduplication alone

### Opportunities

| # | Action | Expected Impact | Effort |
|---|--------|----------------|--------|
| 1 | **Deduplicate V2** | Recover 41.6% of training capacity for fresh examples | Low |
| 2 | **Jargon filter all targets** | Remove examples that teach the model to retain jargon | Low |
| 3 | **FK grade gate (< 8)** | Ensure all targets are patient-readable | Low |
| 4 | **Salvage Asclepius** | Jargon+FK filter could recover ~20K usable sentence pairs | Medium |
| 5 | **Generate more synthetic pairs** | Claude pairs had best quality signal, especially for surgical notes | Medium |
| 6 | **Rebalance to V2 ratio** | 50% terms+phrases, 35% sentences, 15% paragraphs | Low |
| 7 | **Compression ratio gate** | Filter to 0.3 < tgt/src < 1.3 to prevent hallucination | Low |

In [ ]:
# --- Estimate V4 dataset after filters ---
print("ESTIMATED V4 DATASET COMPOSITION")
print("=" * 50)
print()

# V2 after dedup
v2_dedup = int(results["V2 (deployed)"]["n"] * (1 - results["V2 (deployed)"]["dup_pct"] / 100))
print(f"V2 base (deduplicated):        ~{v2_dedup:,}")

# Asclepius salvage estimate (remove 60% via jargon+FK filter)
asclepius_total = 57000  # approximate from file size
asclepius_salvage = int(asclepius_total * 0.35)  # conservative
print(f"Asclepius salvage (filtered):   ~{asclepius_salvage:,}")

# New synthetic
print(f"New Claude synthetic (target):  ~3,000")

total_est = v2_dedup + asclepius_salvage + 3000
print(f"\nEstimated V4 total:            ~{total_est:,}")
print(f"\nTarget granularity:")
print(f"  Terms + phrases: ~{int(total_est * 0.50):,} (50%)")
print(f"  Sentences:       ~{int(total_est * 0.35):,} (35%)")
print(f"  Paragraphs:      ~{int(total_est * 0.15):,} (15%)")